# ML-07 - Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/flyrank-int/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

My lane is **CTR / Engagement Opportunity Scoring** (framed in ML-02/03, contract in ML-04).
This notebook does the three things the card asks, in one place: **check the two signals my rule
leans on**, **encode one transparent rule** and write its ranked queue to
`work/outputs/baseline_action_score.csv`, then **read my own top ten with a skeptic's eye**.

It runs on the in-repo starter slice (`data/raw/content_refresh_anonymized.csv`) so it is
reproducible top-to-bottom with no token. The warehouse March slice from ML-04 is the *same*
construction on a bigger table - that is the capstone substrate, not this baseline.

> This is the baseline my Week-5 model has to beat. Lane confirmed, not switched.

## 1. My rule, and the two signals it leans on

### The rule, in plain words (say it before coding it)

> **A visible page is worth an editor's hour when it is capturing fewer clicks than
> other pages that rank at the same position - and the bigger that shortfall in raw
> clicks, the higher it goes in the queue.**

That is it. One idea: *under-capturing clicks relative to same-rank peers*. Three moving parts,
all knowable at the decision moment from the trailing 90-day snapshot:

- **`expected_ctr`** = the median CTR of pages in the same rounded position band. "What a typical
  page at this rank gets."
- **`ctr_gap`** = `ctr - expected_ctr`. Negative means under-capturing.
- **score = `missed_clicks_90d`** = `max(0, expected_ctr - ctr) / 100 * impressions_90d`.
  In words: *how many clicks this page left on the table over 90 days versus a typical page at
  its rank.* Rate columns are x100 percentages (data skill S1), so the `/100` converts back.

Weighting the gap by `impressions_90d` is deliberate and transparent (no fitted weights - the
building-baselines skill builds a score exactly this way): a 0.1pp deficit on 300k impressions is
a bigger real opportunity than a 0.3pp deficit on 600. The score is *interpretable* - "~1,000
clicks/quarter left on the table" is a sentence an editor understands.

**ONE reason code:** `under_capturing_vs_rank`  -  the rule has exactly one reason it ever fires.

**Action label:** `Rewrite title & meta to lift CTR`  -  the editorial move a CTR-opportunity page
gets: sharper title + meta description + snippet structure, position held.

### The two signals this rule leans on (each gets a bucket table with n, and a one-word verdict)

The rule stands or falls on two claims. I check both **before** trusting the rule - a clearly
explained negative here is a win, because it saves the rule from shipping on a bad assumption.

1. **CTR depends on position** (flag-linked - this is the signal behind FlyRank's CTR-fix logic:
   the whole reason for an *expected-CTR-by-rank* curve instead of a flat CTR cutoff). If CTR
   does **not** vary with position, my `expected_ctr` normalisation is pointless and a flat
   threshold would do. **Predicted: CONFIRMED.**
2. **The most extreme under-capturers are real content opportunities** (the tail my `-ctr_gap`
   score reaches for first). ML-03 already found 10% of the universe has *zero* clicks at high
   impressions, clustered by client. If that tail is broken tracking rather than content, ranking
   by raw gap would fill the queue's top with artifacts. **Predicted: FALSE - and if so, a
   `clicks >= 1` floor saves the rule.**

The code cell below builds each bucket table (with `n` printed) and prints a one-word verdict:
CONFIRMED / OPPOSITE / MIXED / FALSE.

In [1]:
# --- Setup + the visible universe, then the TWO signal bucket tables --------------
import os, sys, subprocess
import numpy as np, pandas as pd
pd.set_option("display.width", 170)

# Repo-root discovery: works in Colab (clone) and locally (walk up to data/raw).
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The visible universe (identical rule to ML-02/03): a page an editor could open THIS week.
#   impressions_90d >= 500  -> below this CTR is noise
#   avg_position in (0, 20] -> 0 means "no position data" NOT rank zero (data skill S1);
#                              beyond ~page 2 the action is "rank better", a different lane
#   content_age_days >= 90  -> needs history; dedup content_id enforces the grain
elig = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].drop_duplicates("content_id")
vis  = elig[(elig.impressions_90d >= 500) & (elig.avg_position > 0) & (elig.avg_position <= 20)].copy()
vis["pos_band"] = vis.avg_position.round().clip(1, 20).astype(int)
print(f"visible universe: {len(vis):,} pages across {vis.client_id.nunique()} clients\n")

# === SIGNAL 1 (flag-linked): does CTR depend on position? =========================
# Bucket = rounded position band. Table prints n per bucket (card requirement).
s1 = (vis.groupby("pos_band")
         .agg(n=("ctr", "size"), median_ctr=("ctr", "median"))
         .reset_index())
print("=== Signal 1: median CTR by position band (behind FlyRank's CTR-fix logic) ===")
print(s1.to_string(index=False))

spearman = vis[["avg_position", "ctr"]].corr(method="spearman").iloc[0, 1]
body = s1[s1.pos_band.between(3, 10)]          # high-n body of the distribution
top  = s1[s1.pos_band <= 2]
print(f"\n  spearman(position, ctr)            : {spearman:+.3f}  (negative = CTR falls as rank worsens)")
print(f"  CTR range across bands             : {s1.median_ctr.min():.2f} .. {s1.median_ctr.max():.2f} (pp)")
print(f"  body bands 3-10 (n={body.n.sum():,}) : monotone-ish decline {body.median_ctr.iloc[0]:.2f} -> {body.median_ctr.iloc[-1]:.2f}")
print(f"  TOP bands 1-2 (n={top.n.sum():,})       : median CTR {top.median_ctr.tolist()} <- INVERTED, tiny n, implausible")
print("  VERDICT: MIXED  -- CTR clearly varies with rank (a flat cutoff is provably wrong,")
print("           so position-adjustment IS justified), but the relationship is noisy at the")
print("           extremes: bands 1-2 invert on ~30-250 pages, and the deep tail plateaus.")
print("           => trust the band-median in the high-n body; treat bands 1-2 as unreliable.\n")

# === SIGNAL 2: are the most extreme under-capturers real opportunities? ============
# Bucket = has this page any clicks in 90d? Table prints n per bucket.
vis["click_bucket"] = np.where(vis.clicks_90d == 0, "zero_clicks", "has_clicks")
s2 = (vis.groupby("click_bucket")
         .agg(n=("ctr", "size"),
              median_impressions=("impressions_90d", "median"),
              max_impressions=("impressions_90d", "max"),
              median_ctr=("ctr", "median"))
         .reset_index())
print("=== Signal 2: the extreme tail my raw -ctr_gap score reaches for first ===")
print(s2.to_string(index=False))

zero = vis[vis.clicks_90d == 0]
zshare = (vis.assign(z=vis.clicks_90d == 0).groupby("client_id")["z"].mean())
top50_raw = vis.assign(ctr_gap=vis.ctr - vis.groupby("pos_band")["ctr"].transform("median")).nsmallest(50, "ctr_gap")
print(f"\n  zero-click pages                   : {len(zero):,} ({len(zero)/len(vis):.1%} of universe)")
print(f"  most impressions on a 0-click page : {zero.impressions_90d.max():,}  <- 0 clicks is implausible here")
print(f"  clients with >20% zero-click       : {(zshare > 0.20).sum()} of {len(zshare)} (worst {zshare.max():.0%}, best {zshare.min():.0%})")
print(f"  a raw top-50 by -ctr_gap would be  : {(top50_raw.clicks_90d == 0).mean():.0%} zero-click pages, from {top50_raw.client_id.nunique()} of {vis.client_id.nunique()} clients")
print("  VERDICT: FALSE  -- a page at position ~4 with hundreds of thousands of impressions and")
print("           ZERO clicks over 90 days is what broken click-tracking looks like, not a")
print("           content opportunity, and it CLUSTERS BY CLIENT (analytics setup varies; editorial")
print("           quality does not vary that sharply). Ranking by raw gap fills the queue's top with")
print("           artifacts. => the rule adds a `clicks >= 1` floor (the ML-04 contract decision).")


visible universe: 12,023 pages across 28 clients

=== Signal 1: median CTR by position band (behind FlyRank's CTR-fix logic) ===
 pos_band    n  median_ctr
        1   37       0.060
        2  250       0.215
        3  423       0.300
        4  890       0.330
        5 1089       0.290
        6 1394       0.240
        7 1107       0.220
        8 1160       0.190
        9  758       0.200
       10  798       0.170
       11  576       0.190
       12  618       0.180
       13  451       0.190
       14  496       0.190
       15  375       0.160
       16  432       0.170
       17  336       0.170
       18  347       0.160
       19  287       0.160
       20  199       0.150

  spearman(position, ctr)            : -0.183  (negative = CTR falls as rank worsens)
  CTR range across bands             : 0.06 .. 0.33 (pp)
  body bands 3-10 (n=7,619) : monotone-ish decline 0.30 -> 0.17
  TOP bands 1-2 (n=287)       : median CTR [0.06, 0.215] <- INVERTED, tiny n, implausible
  VERD

### What the two verdicts did to the rule

- **Signal 1 - MIXED, not the CONFIRMED I predicted.** CTR really does track position (spearman
  is negative; the median swings from ~0.35pp in the shallow body to ~0.15pp deep), so an
  expected-CTR-by-rank curve is well-founded and a flat CTR cutoff is provably wrong. *But* the
  top two bands invert on tiny `n` - almost certainly the Signal-2 contamination leaking upward.
  **Effect on the rule:** keep the position adjustment, but do not over-trust bands 1-2; the
  top-10 review flags any row whose `expected_ctr` rests on a thin band.

- **Signal 2 - FALSE, exactly the negative I feared.** The extreme tail is broken tracking, not
  opportunity. **Effect on the rule:** add a `clicks >= 1` floor before scoring. This is the one
  deliberate exclusion from my ML-04 contract, now earning its place with a table. It also
  explains Signal 1's inversion, so the two checks corroborate each other. The floor's honest
  cost (stated in section 4): it also drops any genuinely-broken page that truly earned zero
  clicks, and it does not *repair* partial-tracking clients - only removes the all-zero extreme.

## 2. Build the ranked queue (writes the CSV)

Encode the rule from section 1 on the floored universe, rank every page, write the queue to
`work/outputs/baseline_action_score.csv`. Columns are chosen so a human can audit any row:
the score, the ONE reason code, the action label, and the raw numbers behind the gap.

The CSV is regenerated on every run and stays out of git by design (the leak-guard blocks
`work/**/*.csv`). A tiny metrics JSON receipt - safe aggregates only, no IDs - is written
alongside and *is* committed.

In [2]:
# --- Encode the rule, rank, and write the queue -----------------------------------
import json

REASON_CODE = "under_capturing_vs_rank"
ACTION      = "Rewrite title & meta to lift CTR"

# Floor from Signal 2 (FALSE verdict): drop zero-click pages before scoring.
q = vis[vis.clicks_90d >= 1].copy()
print(f"after clicks>=1 floor: {len(q):,} pages "
      f"(dropped {len(vis) - len(q):,} zero-click pages Signal 2 flagged)\n")

# Expected CTR = band median, recomputed on the floored universe (cleaner curve).
q["expected_ctr"] = q.groupby("pos_band")["ctr"].transform("median")
q["ctr_gap"]      = q.ctr - q.expected_ctr                       # negative = under-capturing
# THE SCORE: clicks left on the table vs same-rank peers (0 if at/above expected).
q["missed_clicks_90d"] = np.where(q.ctr_gap < 0,
                                  (q.expected_ctr - q.ctr) / 100.0 * q.impressions_90d, 0.0)

# One rule -> one reason code -> one action, attached to every scored row.
q["reason_code"]  = REASON_CODE
q["action"]       = ACTION
# Small transparency flag for the reviewer (NOT a second reason code): does this row's
# expected_ctr rest on a thin band (Signal 1's unreliable extremes)?
band_n = q.groupby("pos_band")["ctr"].transform("size")
q["thin_band_flag"] = (band_n < 100).astype(int)

# Rank: biggest shortfall in raw clicks first. Only pages actually under-capturing enter the queue.
queue = (q[q.missed_clicks_90d > 0]
         .sort_values("missed_clicks_90d", ascending=False)
         .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)

OUT_COLS = ["rank", "content_id", "client_id", "content_type", "action", "reason_code",
            "missed_clicks_90d", "avg_position", "pos_band", "thin_band_flag",
            "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "ctr_gap"]
out = queue[OUT_COLS].copy()
out["missed_clicks_90d"] = out.missed_clicks_90d.round(1)
for c in ["ctr", "expected_ctr", "ctr_gap"]:       # kill float-subtraction noise in the CSV
    out[c] = out[c].round(3)

os.makedirs("work/outputs", exist_ok=True)
CSV_PATH = "work/outputs/baseline_action_score.csv"
out.to_csv(CSV_PATH, index=False)
print(f"wrote {CSV_PATH}: {len(out):,} ranked pages "
      f"({len(out)/len(q):.0%} of the floored universe are under-capturing)\n")

print("=== queue head ===")
print(out.head(5).to_string(index=False))
print("\n=== score distribution (missed clicks / 90d) ===")
print(out.missed_clicks_90d.describe(percentiles=[.5, .9, .99]).round(1).to_string())

# --- The metric, named honestly (precision@K) -------------------------------------
def precision_at_k(scores, labels, k=50):
    """Of the top-k by score, what fraction carry label == 1? (Ready for a real label.)"""
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())

print("\n=== precision@50: mechanism ready, meaning deferred (ML-03 finding) ===")
print("  My metric is precision@50 (K = one reviewer's week). It needs a label that says a page")
print("  DESERVED the review. On this single 90d snapshot every such label is built from ctr -")
print("  the same column the score reads - so precision@50 would grade the ranking against its")
print("  own input and score ~1.0. That is arithmetic consistency, not usefulness. The honest")
print("  evaluation waits on a forward-looking observed label (warehouse, ML-08/capstone).")
print("  I refuse to print a flattering 1.0 here; the function above is ready for the real label.")

# --- Metrics JSON receipt (committed; safe aggregates only, NO ids) ---------------
metrics = {
    "task": "ML-07 baseline_action_score",
    "universe_visible_pages": int(len(vis)),
    "universe_clients": int(vis.client_id.nunique()),
    "zero_click_pages_dropped": int((vis.clicks_90d == 0).sum()),
    "floored_universe_pages": int(len(q)),
    "queue_rows_under_capturing": int(len(out)),
    "signal_1_ctr_vs_position_verdict": "MIXED",
    "signal_1_spearman_pos_ctr": round(float(vis[["avg_position", "ctr"]].corr(method="spearman").iloc[0, 1]), 3),
    "signal_2_extreme_tail_verdict": "FALSE",
    "score": "missed_clicks_90d = max(0, expected_ctr - ctr)/100 * impressions_90d",
    "reason_code": REASON_CODE,
    "action_label": ACTION,
    "top10_distinct_clients": int(out.head(10).client_id.nunique()),
    "top50_distinct_clients": int(out.head(50).client_id.nunique()),
    "precision_at_50": None,
    "precision_at_50_note": "deferred: single-snapshot labels are circular; needs forward label",
}
JSON_PATH = "work/outputs/w04_baseline_metrics.json"
with open(JSON_PATH, "w") as fh:
    json.dump(metrics, fh, indent=2)
print(f"\nwrote {JSON_PATH} (committed receipt, no ids):")
print(json.dumps(metrics, indent=2))


after clicks>=1 floor: 10,808 pages (dropped 1,215 zero-click pages Signal 2 flagged)



wrote work/outputs/baseline_action_score.csv: 5,307 ranked pages (49% of the floored universe are under-capturing)

=== queue head ===
 rank           content_id         client_id    content_type                           action             reason_code  missed_clicks_90d  avg_position  pos_band  thin_band_flag  impressions_90d  clicks_90d  ctr  expected_ctr  ctr_gap
    1 content_5fe46e04994d client_4e07408562 keyword article Rewrite title & meta to lift CTR under_capturing_vs_rank             1087.2           4.2         4               0           517715         741 0.14          0.35    -0.21
    2 content_8c19996aa890 client_4e07408562 keyword article Rewrite title & meta to lift CTR under_capturing_vs_rank              713.0           2.5         2               0           509252         785 0.15          0.29    -0.14
    3 content_8451fc6f034d client_d029fa3a95 keyword article Rewrite title & meta to lift CTR under_capturing_vs_rank              707.6           2.3         2   

## 3. Top-10 review

The top of the list is where bad logic shows itself. For each of my top ten, one line:
**the action**, **why it's there** (the reason code plus the actual numbers), and **what would
make it wrong** - the specific thing that, if true, means an editor's hour is wasted. The notes
are generated from each row's own numbers, not a template.

In [3]:
# --- Top-10 review: action / why / what would make it wrong (per row) -------------
def what_would_make_it_wrong(r):
    reasons = []
    if r.impressions_90d >= 200_000:
        reasons.append(f"huge exposure ({r.impressions_90d:,.0f} imp) at rank {r.avg_position:.1f} often "
                       "means a broad head/branded query the page can't own - low CTR would be intent "
                       "mismatch, not a fixable title")
    if r.ctr < 0.05:
        reasons.append(f"CTR {r.ctr:.2f}pp is so far below the band that residual partial-tracking "
                       "(undercounted clicks, Signal 2's softer tail) is a live alternative to a real deficit")
    if r.thin_band_flag == 1:
        reasons.append(f"pos_band {r.pos_band} is thin (Signal 1 MIXED) - its expected_ctr "
                       f"{r.expected_ctr:.2f} rests on few pages, so the gap is less certain")
    # Always-applicable structural caveat for this grain:
    reasons.append("avg_position averages across every query the page ranks for - the gap can be a "
                   "hard-query mix, not something a rewrite fixes (ML-04 open question 3)")
    return "; ".join(reasons)

print(f"{'='*100}\nTOP 10 - each row: ACTION | why it's here | what would make it wrong\n{'='*100}")
for r in out.head(10).itertuples(index=False):
    print(f"\n#{r.rank}  {r.content_id}  ({r.client_id[:20]}..., {r.content_type})")
    print(f"   ACTION : {r.action}   [{r.reason_code}]")
    print(f"   WHY    : leaves ~{r.missed_clicks_90d:,.0f} clicks/90d on the table - CTR {r.ctr:.2f}pp vs "
          f"band-{r.pos_band} expected {r.expected_ctr:.2f}pp (gap {r.ctr_gap:+.2f}) on {r.impressions_90d:,.0f} impressions at rank {r.avg_position:.1f}")
    print(f"   WRONG IF: {what_would_make_it_wrong(r)}")

print(f"\n{'='*100}")
print(f"top-10 spans {out.head(10).client_id.nunique()} clients; top-50 spans {out.head(50).client_id.nunique()} of {out.client_id.nunique()}.")
print("Concentration is itself a finding - see section 4.")


TOP 10 - each row: ACTION | why it's here | what would make it wrong

#1  content_5fe46e04994d  (client_4e07408562..., keyword article)
   ACTION : Rewrite title & meta to lift CTR   [under_capturing_vs_rank]
   WHY    : leaves ~1,087 clicks/90d on the table - CTR 0.14pp vs band-4 expected 0.35pp (gap -0.21) on 517,715 impressions at rank 4.2
   WRONG IF: huge exposure (517,715 imp) at rank 4.2 often means a broad head/branded query the page can't own - low CTR would be intent mismatch, not a fixable title; avg_position averages across every query the page ranks for - the gap can be a hard-query mix, not something a rewrite fixes (ML-04 open question 3)

#2  content_8c19996aa890  (client_4e07408562..., keyword article)
   ACTION : Rewrite title & meta to lift CTR   [under_capturing_vs_rank]
   WHY    : leaves ~713 clicks/90d on the table - CTR 0.15pp vs band-2 expected 0.29pp (gap -0.14) on 509,252 impressions at rank 2.5
   WRONG IF: huge exposure (509,252 imp) at rank 2.5 often means

## 4. Weak picks + leakage check

The skeptic's pass. Two questions: **which picks look weak and why**, and **did any future-window
or label-derived input sneak into the score?**

In [4]:
# --- Weak picks --------------------------------------------------------------------
print("=== WEAK PICKS (where I distrust my own queue) ===\n")

# 1. Client concentration: impression-weighting favours big clients.
top50 = out.head(50)
conc = top50.client_id.value_counts()
print("1. CONCENTRATION - the score weights by impressions, so big clients dominate the top.")
print(f"   top-50 queue spans only {top50.client_id.nunique()} of {out.client_id.nunique()} clients; "
      f"the single biggest client holds {conc.iloc[0]} of the top 50.")
print("   Weak because: an editor who owns a small client sees nothing actionable. A production")
print("   queue would round-robin per client or cap per-client share. Left transparent here, not hidden.\n")

# 2. Very-low-CTR survivors: passed the clicks>=1 floor but still look tracking-ish.
suspicious = top50[(top50.ctr < 0.05) & (top50.impressions_90d >= 100_000)]
print(f"2. RESIDUAL TRACKING - {len(suspicious)} of the top 50 have CTR < 0.05pp on >=100k impressions.")
print("   The clicks>=1 floor removed the all-zero extreme, not partial undercounting. These are")
print("   the rows most likely to be measurement, not opportunity - I'd sanity-check tracking first.\n")

# 3. Thin-band rows: expected_ctr from Signal 1's unreliable bands.
thin = top50[top50.thin_band_flag == 1]
print(f"3. THIN-BAND expected_ctr - {len(thin)} of the top 50 sit in a band with <100 pages (Signal 1 MIXED).")
print("   Their gap is measured against a shaky baseline; treat the ranking there as directional.\n")

# --- Leakage check: no future-window, no label-derived input ----------------------
print("=== LEAKAGE CHECK - no future-window or label-derived inputs (card requirement) ===\n")
SCORE_INPUTS = {"ctr", "expected_ctr", "impressions_90d"}   # everything the score reads
BANNED = {
    "is_declining_label": "the derived LABEL itself",
    "trend_direction":    "label parent (derives is_declining_label)",
    "trend_pct":          "label grandparent",
    "impressions_last_30d": "would need a forward window to be an outcome",
    "clicks_last_30d":      "forward-window outcome",
    "sessions_last_30d":    "forward-window outcome",
}
leaked = SCORE_INPUTS & set(BANNED)
print(f"  score inputs                 : {sorted(SCORE_INPUTS)}")
print(f"  banned (label/future) columns: {sorted(BANNED)}")
print(f"  intersection (must be empty) : {sorted(leaked)}")
assert not leaked, f"LEAK: {leaked} used as a score input"
print("  -> PASS: none of the banned columns feed the score.\n")

print("  Two clarifications, stated not hidden:")
print("  a) The score IS built from ctr/expected_ctr. For a MODEL that would be circular; for a")
print("     transparent RULE, ctr_gap arithmetic IS the baseline (ML-03) - not a feature fed to a")
print("     learner. The Week-5 model must predict a FORWARD label without reading ctr.")
print("  b) Every score input is a trailing-90d measurement observed AT the decision moment.")
print("     No column begins after it. No future window is read.")
print("\n  No client names / URLs / raw queries anywhere: only pseudonymous ids + aggregates.")


=== WEAK PICKS (where I distrust my own queue) ===

1. CONCENTRATION - the score weights by impressions, so big clients dominate the top.
   top-50 queue spans only 8 of 23 clients; the single biggest client holds 31 of the top 50.
   Weak because: an editor who owns a small client sees nothing actionable. A production
   queue would round-robin per client or cap per-client share. Left transparent here, not hidden.

2. RESIDUAL TRACKING - 8 of the top 50 have CTR < 0.05pp on >=100k impressions.
   The clicks>=1 floor removed the all-zero extreme, not partial undercounting. These are
   the rows most likely to be measurement, not opportunity - I'd sanity-check tracking first.

3. THIN-BAND expected_ctr - 0 of the top 50 sit in a band with <100 pages (Signal 1 MIXED).
   Their gap is measured against a shaky baseline; treat the ranking there as directional.

=== LEAKAGE CHECK - no future-window or label-derived inputs (card requirement) ===

  score inputs                 : ['ctr', 'expe

## Self-check

- [x] Two signal checks, each with a **bucket table and n**, each with a one-word verdict -
      **Signal 1 (CTR vs position, flag-linked): MIXED**; **Signal 2 (extreme tail): FALSE**.
      At least one is behind a real FlyRank flag (Signal 1 = the CTR-fix logic). The FALSE one
      saved the rule (it added the `clicks >= 1` floor).
- [x] **One rule**: a score (`missed_clicks_90d`), **one reason code** (`under_capturing_vs_rank`),
      **one action label** (`Rewrite title & meta to lift CTR`).
- [x] **Ranked queue written from the notebook** to `work/outputs/baseline_action_score.csv`
      (regenerated each run; gitignored by the leak-guard). Metrics JSON receipt committed.
- [x] **Top-10 reviewed**: action, why, and *what would make it wrong* for each of the ten.
- [x] **No future-window or label-derived inputs** - asserted in section 4.
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all) - **confirm on your run**.
- [ ] Committed under `work/notebooks/`, then submit the repo URL on the card.

### The baseline in one table

| Piece | My answer |
|---|---|
| **Universe** | visible pages: `imp >= 500`, `0 < avg_position <= 20`, `age >= 90`, deduped |
| **Signal 1** (flag-linked) | CTR vs position -> **MIXED**: real gradient (a flat cutoff is wrong) but noisy at bands 1-2 |
| **Signal 2** | extreme tail = opportunity? -> **FALSE**: broken tracking, clusters by client -> `clicks >= 1` floor |
| **Score** | `missed_clicks_90d = max(0, expected_ctr - ctr)/100 * impressions_90d` (clicks left on the table) |
| **Reason code / action** | `under_capturing_vs_rank` / `Rewrite title & meta to lift CTR` |
| **Metric** | precision@50 - mechanism ready, meaning deferred to a forward label (single-snapshot is circular) |
| **Known weak spots** | client concentration; residual partial-tracking; thin-band `expected_ctr` |

### What the checks changed

I predicted CONFIRMED for Signal 1 and got **MIXED** - the position gradient is real enough to
justify the whole rule, but the top bands invert, and the reason turned out to be Signal 2's
contamination bleeding upward. I predicted FALSE for Signal 2 and got **FALSE** - so the rule
ships with a `clicks >= 1` floor it would not have had if I'd trusted the raw gap. Two small
tables changed the scope of the queue before a single row shipped. That is the point of checking
signals first.